# Matchmaking System

We want to design a new matchmaking system for the AI Arena StarCraft 2 bot ladder **without rounds** and **without divisions** (divisions may still be present in the frontend to report progress, but are not used for matchmaking itself).

The matchmaking must ensure that **all** bots

- play a "fair" number of matches;
- play predominantly matches against opponents of a similar skill;
- play matches against varied opponents within those of similar skill;
- play matches which are somewhat evenly distributed in time.

We verify these requirements by running the matchmaker on a [model of the real ladder](ladder_model.ipynb) and checking the match frequency distributions, rating-difference distributions, and opponent distributions.

## Scoring Function

The matchmaker is **greedy**: whenever a game slot becomes available, it scores all pairs of idle bots and starts the match with the highest score. The score for a candidate match between bots A and B is

$$
\text{score}(A, B) = w_\text{skill} \cdot f_\text{skill}(A, B) + w_\text{fair} \cdot f_\text{fair}(A, B) + w_\text{var} \cdot f_\text{var}(A, B) + w_\text{time} \cdot f_\text{time}(A, B),
$$

where the components correspond to the four matchmaking requirements:

### Skill Matching

Matches between similarly-rated bots should be preferred. We use a Gaussian decay over the rating difference:

$$
f_\text{skill}(A, B) = \exp\!\left(-\frac{(R_A - R_B)^2}{2\tau^2}\right),
$$

where $\tau$ controls how tolerant the system is of rating differences. Values in $[0, 1]$.

### Fairness

Bots that have played fewer games should be prioritised. Let $g_i$ be the number of games bot $i$ has completed in the last 24 hours and $\bar{g}$ the mean across all currently active bots:

$$
f_\text{fair}(A, B) = \max(\bar{g} - g_A,\; \bar{g} - g_B).
$$

Using the max preserves the sign of the deficit: it is positive when at least one bot is underplayed and negative when both are overplayed. The most underplayed bot drives the score, while the partner is chosen by the other components. Bots that recently joined the ladder naturally receive a temporary priority boost since their game count in the window is low.

### Opponent Variety

Repeated match-ups should be discouraged. Let $a_{AB}$ be the number of games A has played since last facing B, and $b_{AB}$ the same for B ($a_{AB} = b_{AB} = \infty$ if they have never met):

$$
f_\text{var}(A, B) = 1 - \exp\!\left(-\frac{\sqrt{a_{AB} \cdot b_{AB}}}{\lambda}\right),
$$

where $\lambda$ controls how quickly a past match-up is "forgotten". The geometric mean $\sqrt{a_{AB} \cdot b_{AB}}$ gives partial credit when one bot has diversified a lot but the other hasn't, while still requiring both to have moved on. Values in $[0, 1]$.

### Time Distribution

Bots that have been idle longer should be preferred. Let $t_i$ be the time since bot $i$ last finished a game. We use the RMS (root mean square) to aggregate the per-bot idle times:

$$
f_\text{time}(A, B) = \sqrt{\frac{t_A^2 + t_B^2}{2}}.
$$

The RMS is pulled toward the longer idle time, ensuring that no bot is left idle for too long, while still giving partial credit when both bots have been waiting. Idle times are always non-negative, so the sign issue that affects fairness does not apply here.

### Parameters

| Parameter | Role |
|-----------|------|
| $w_\text{skill},\, w_\text{fair},\, w_\text{var},\, w_\text{time}$ | Relative importance of each objective |
| $\tau$ | Rating-difference tolerance |
| $\lambda$ | Opponent-variety decay rate |

Since the components have different scales, the weights absorb normalisation — they are not directly comparable across components.